# Total Site
This notebook compares process and site-level targets for a pulp mill, then shows how total-site graphing and cogeneration results change the utility-system interpretation.


## Case context
The packaged `pulp_mill.json` case represents a multi-area pulp mill with steam levels, hot-water utility interaction, and site-wide utility tradeoffs. The workflow is: solve the baseline problem, compare direct, total-process, and total-site targets, inspect the process GCC against the total-site plots, and then run cogeneration targeting on the total-site target.


In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory

import pandas as pd
import plotly.graph_objects as go
from IPython.display import HTML, display

from OpenPinch import PinchProblem
from OpenPinch.lib.enums import *
from OpenPinch.resources import copy_sample_case

PLOT_WIDTH = 720
PLOT_HEIGHT = 540


def display_plotly(fig, *, width: int = PLOT_WIDTH, height: int = PLOT_HEIGHT):
    fig = fig.to_dict()
    figure = go.Figure(fig)
    figure.update_layout(width=width, height=height, autosize=False)
    display(HTML(figure.to_html(full_html=False, include_plotlyjs="cdn")))


workspace = TemporaryDirectory()
work_dir = Path(workspace.name)
case_path = copy_sample_case("pulp_mill.json", work_dir / "pulp_mill.json")

In [ ]:
problem = PinchProblem(problem_filepath=case_path)
problem.target()
TS_targets = problem.master_zone.targets[TargetType.TS.value]
bleach_plant = problem.master_zone.subzones["Bleaching"]
bleach_target = bleach_plant.targets[TargetType.DI.value]

print(bleach_target.hot_utility_target)

In [ ]:
summary = problem.summary_frame()

target_order = [
    "pulp_mill/Direct Integration",
    "pulp_mill/Total Process Target",
    "pulp_mill/Total Site Target",
]
target_labels = {
    "pulp_mill/Direct Integration": "Direct Integration",
    "pulp_mill/Total Process Target": "Total Process Target",
    "pulp_mill/Total Site Target": "Total Site Target",
}
target_rows = summary.loc[
    summary["Target"].isin(target_order),
    [
        "Target",
        "Hot Utility Target",
        "Cold Utility Target",
        "Heat Recovery",
        "Hot Pinch",
        "Cold Pinch",
    ],
].copy()
target_rows["Target Level"] = target_rows["Target"].map(target_labels)
target_rows = target_rows.set_index("Target").loc[target_order].reset_index(drop=True)
target_rows

## Inspect these outputs first
Start with the three-row summary table, then compare the metric shifts visually. This answers whether moving from direct integration to total-site targeting materially changes the utility picture before looking at the graph shapes.


In [ ]:
target_metrics = target_rows.melt(
    id_vars="Target Level",
    value_vars=["Hot Utility Target", "Cold Utility Target", "Heat Recovery"],
    var_name="Metric",
    value_name="Value",
)

site_targets_fig = go.Figure()
for metric in ["Hot Utility Target", "Cold Utility Target", "Heat Recovery"]:
    metric_rows = target_metrics.loc[target_metrics["Metric"] == metric]
    site_targets_fig.add_bar(
        name=metric,
        x=metric_rows["Target Level"],
        y=metric_rows["Value"],
        text=[f"{value:,.0f}" for value in metric_rows["Value"]],
        textposition="outside",
    )

site_targets_fig.update_layout(
    barmode="group",
    title="Utility and recovery comparison across target levels",
    xaxis_title="Target level",
    yaxis_title="kW",
)
display_plotly(site_targets_fig, height=620)

## Graphs to compare
Use the total-site profiles and the Site Utility Grand Composite Curve to understand site-wide utility interaction and steam-system leverage.


In [ ]:
total_site_profiles = problem.plot(graph_type="Total Site Profiles")
display_plotly(total_site_profiles)

In [ ]:
sugcc = problem.plot(graph_type="Site Utility Grand Composite Curve")
display_plotly(sugcc)

## Total site cogeneration screen
The cogeneration screen should be run from the total-site target because that is the level where utility-system interactions and steam-pressure opportunities have already been reconciled.


In [ ]:
cogen_bleach = problem.target.cogeneration(zone_name="Bleaching")
print(cogen_bleach.work_target)
print(cogen_bleach.turbine_efficiency_target)

In [ ]:
cogeneration_target = problem.target.cogeneration(
    options={
        "DO_TURBINE_WORK": True,
        "base_target_type": "Total Site Target",
    }
)

cogeneration_summary = pd.DataFrame(
    [
        {
            "Target": cogeneration_target.name,
            "Hot Utility Target": cogeneration_target.hot_utility_target,
            "Cold Utility Target": cogeneration_target.cold_utility_target,
            "Heat Recovery": cogeneration_target.heat_recovery_target,
            "Hot Pinch": cogeneration_target.hot_pinch,
            "Cold Pinch": cogeneration_target.cold_pinch,
            "Power Cogeneration Target": cogeneration_target.work_target,
            "Turbine Efficiency Target": cogeneration_target.turbine_efficiency_target,
        }
    ]
)
cogeneration_summary

## Interpretation
In this pulp mill, the direct GCC helps explain local process bottlenecks, but the total-site plots show the more important system decision: how steam and hot-water utility interaction changes the minimum utility picture once the whole site is reconciled. The SUGCC is the graph to read when deciding whether utility-system changes, including cogeneration, are worth studying further. The cogeneration result should be interpreted as a total-site opportunity screen, not as a process-unit conclusion.
